In [67]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [68]:
df=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\ratings.csv")

In [69]:
movies_meta=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\movies_metadata.csv")

C:\Users\kirti\AppData\Local\Temp\ipykernel_5116\1612413770.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_meta=pd.read_csv(r"C:\Users\kirti\Desktop\AI Engineer\End-To-End Project\Movie Recommendation System\data\movies_metadata.csv")


In [70]:
df.head()

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


In [71]:
movies_meta=movies_meta.rename(columns={"id":"movieId"})
movies_meta.head()

,adult,belongs_to_collection,budget,genres,homepage,movieId,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [72]:
movies_meta["movieId"]=pd.to_numeric(movies_meta["movieId"],errors="coerce") ## we use to_numeric() when we have to handle errors in data, .astype() throws error (only use when clean)

In [73]:
print(df.shape[0],movies_meta.shape[0])

26024289 45466


In [74]:
merg_df=pd.merge(df,movies_meta, on="movieId",how="inner") ## as i'll only need table that has both movie id and its name

In [75]:
merg_df.columns

Index(['userId', 'movieId', 'rating', 'timestamp', 'adult',
       'belongs_to_collection', 'budget', 'genres', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title', 'video', 'vote_average', 'vote_count'],
      dtype='object')

In [76]:
fdf=merg_df[['userId', 'movieId', 'rating', 'timestamp', 'title']]
fdf

,userId,movieId,rating,timestamp,title
0,1,110,1.0,1425941529,Three Colors: Red
1,1,147,4.5,1425942435,The 400 Blows
2,1,858,5.0,1425941523,Sleepless in Seattle
3,1,1246,5.0,1425941556,Rocky Balboa
4,1,1968,4.0,1425942148,Fools Rush In
...,...,...,...,...,...
11437632,270896,48780,5.0,1257031830,Boat
11437633,270896,49530,4.0,1257034436,In Time
11437634,270896,54001,4.0,1257034331,The Traveler
11437635,270896,54503,4.0,1257033886,The Mystery of Chess Boxing


In [77]:
## fdf.pivot_table(index="userId",columns="movieId",values="rating")        Pivot table didn't worked because too many movies and 90 % + are sparse.

In [78]:
from scipy.sparse import csr_matrix ## Compressed sparse rows
csr_matrix=csr_matrix((fdf["rating"],(fdf["userId"],fdf["movieId"])))
## Sparse matrix only stores non-zero values to save space and be memmory efficient.

Stored in this format:

- data = [3,2,1,4]
- indices = [4,2,3,6]
- indptr = [0,1,3,4]

Row 0 : 

data[0:1] = 3 rating

indices[0:1] = 4 th column

Row 1 : 

data[1:3] = 2,1 rating

indices[1:3] = 2nd, 3rd column

In [79]:
rows,cols=csr_matrix.nonzero()
print(rows,"\n", cols)

[     1      1      1 ... 270896 270896 270896] 
 [  110   147   858 ... 54001 54503 58559]


In [81]:
rows.shape

(11436568,)

In [82]:
cols.shape

(11436568,)

In [83]:
test_size=round(0.2*csr_matrix.shape[0])
test_size

54179

In [87]:
test_rows=rows[test_idx]    ## Test-set indexes
test_cols=cols[test_idx]

In [88]:
train_matrix=csr_matrix.copy()

In [89]:
train_matrix[test_rows,test_cols]=0
train_matrix.eliminate_zeros() ## removes zeros from storage

In [138]:
train_matrix.shape

(270897, 176274)

In [145]:
np.array(train_matrix.sum(axis=1)).flatten()/ train_matrix.getnnz(axis=1)

C:\Users\kirti\AppData\Local\Temp\ipykernel_5116\3405331463.py:1: RuntimeWarning: invalid value encountered in divide
  np.array(train_matrix.sum(axis=1)).flatten()/ train_matrix.getnnz(axis=1)


array([       nan, 4.1       , 3.        , ..., 2.56034483, 4.23529412,
       3.96564885], shape=(270897,))

- SVD does decompostion of matrix into smaller matrix of latent feature which has least reconstruction error.
- Original matrix = latent matrix (user) x latent matrix (movie)
- Truncated SVD uses only top_n latent features to reconstruct matrix for better generalization and faster prediction.

In [90]:
from sklearn.decomposition import TruncatedSVD ## Creates two matrix from single matrix having same latent features.
model=TruncatedSVD(n_components=50)
user_latent=model.fit_transform(train_matrix)  ## User latent features

Matrix Factorization idea shows up in:
- PCA
- SVD
- Word embeddings (Tf-idf Vectorizer)
- Transformers (attention factorization idea)

In [91]:
movie_latent=model.components_.T   ## Movie latent features (Transpose because reconstruction made movie latent transpose)

In [92]:
print("User latent:",user_latent.shape,"Movie_latent",movie_latent.shape)

User latent: (270897, 50) Movie_latent (176274, 50)


Why cosine be better sometimes rather than dot product?

In [104]:
user_latent[test_rows[0]] @ movie_latent[test_cols[0]]

np.float64(1.146207858921269)

In [114]:
y_pred=[]
for i,j in zip(test_rows,test_cols):
    pred=user_latent[i] @ movie_latent[j]
    y_pred.append(pred)

In [115]:
y_test=[]
for i,j in zip(test_rows,test_cols):
    pred=csr_matrix[i,j]
    y_test.append(pred)

In [123]:
y_test

[np.float64(4.0),
 np.float64(1.0),
 np.float64(3.0),
 np.float64(2.5),
 np.float64(2.0),
 np.float64(4.5),
 np.float64(3.0),
 np.float64(5.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(4.0),
 np.float64(3.0),
 np.float64(3.0),
 np.float64(3.5),
 np.float64(3.0),
 np.float64(4.5),
 np.float64(3.5),
 np.float64(0.5),
 np.float64(4.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(4.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(3.0),
 np.float64(3.0),
 np.float64(3.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(3.0),
 np.float64(1.5),
 np.float64(3.5),
 np.float64(3.0),
 np.float64(5.0),
 np.float64(2.5),
 np.float64(4.0),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(4.0),
 np.float64(0.5),
 np.float64(4.0),
 np.float64(3.0),
 np.float64(1.0),
 np.float64(4.0),
 np.float64(3.5),
 np.float64(2.5),
 np.float64(5.0),
 np.float64(4.0),
 np.float64(5.0),
 np.float64(4.5),
 np.float64(4.5),
 np.float64(3.0),
 np.float64(4.0),
 np.float64(4.5),
 np.float64(3.5),
 np.float6

In [122]:
y_pred

[np.float64(1.146207858921269),
 np.float64(0.34440280304310367),
 np.float64(0.7843384498554314),
 np.float64(0.497588944774029),
 np.float64(1.037298190937788),
 np.float64(1.3304161224799416),
 np.float64(0.09168105193659626),
 np.float64(0.9393488817222649),
 np.float64(3.041384354829224),
 np.float64(1.945162684170373),
 np.float64(2.4733371550088847),
 np.float64(0.2987128870797286),
 np.float64(1.1420133123596545),
 np.float64(0.3727283800125857),
 np.float64(1.3023935195671639),
 np.float64(2.0340739373593637),
 np.float64(1.1243860644719803),
 np.float64(-0.03819899926513571),
 np.float64(0.8989516814141172),
 np.float64(0.03811957906851045),
 np.float64(0.9466064909384847),
 np.float64(1.0627289618460414),
 np.float64(2.3991448984674717),
 np.float64(1.0814824559639604),
 np.float64(1.7293312931985463),
 np.float64(3.389555405359352),
 np.float64(0.3386300244100485),
 np.float64(2.1907729791088357),
 np.float64(3.0265914105413088),
 np.float64(0.006093319714296978),
 np.float

In [129]:
from sklearn.metrics import mean_squared_error
mse=mean_squared_error(y_test,y_pred)
rmse=np.sqrt(mse)

In [131]:
print("Mse:",mse,"Rmse:",rmse)

Mse: 8.30217829172297 Rmse: 2.8813500814241526


In [ ]:
y

In [98]:
csr_matrix[test_rows,test_cols]

matrix([[4. , 1. , 3. , ..., 1. , 4. , 4.5]], shape=(1, 54179))

In [95]:
user_latent @ movie_latent

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 176274 is different from 50)

In [94]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix=cosine_similarity(user_latent,movie_latent)

MemoryError: Unable to allocate 356. GiB for an array with shape (270897, 176274) and data type float64